# Lesson 3 — Optimizers: SGD, Momentum and Adam

So far every training loop ended with

```python
for p in model.parameters():
    p.data -= learning_rate * p.grad
```

Backprop tells us the **direction** that reduces the loss. *How far* to step, and whether to remember past steps, is a
separate design decision: the **optimizer**. In this lesson you implement the three optimizers that cover most of deep learning
practice, watch them on a 2-D loss surface, and race them on a real network.

**You will learn to**
1. Separate *model*, *autograd* and *optimizer* behind a `step()` / `zero_grad()` API (just like PyTorch)
2. Implement SGD, SGD with momentum, and Adam
3. Understand the largest stable learning rate, and why momentum and Adam help in ravines

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so `import nanograd` works from the lessons/ folder without installing

def check(name, test):
    """ run a test function; print a tick, a cross, or a reminder if not implemented yet """
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet")
        return
    except Exception as e:
        print(f"❌ {name}: raised {type(e).__name__}: {e}")
        return
    print(f"{'✅' if ok else '❌'} {name}")

In [ ]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
from nanograd.engine import Value
from nanograd.optim import Optimizer

## Part 1 — A ravine

Real loss surfaces are often much steeper in some directions than others. Here is the simplest example — an elongated bowl:

$$f(x, y) = 0.5\,x^2 + 10\,y^2$$

The minimum is at (0, 0). It is 20× more curved along `y` than along `x`.

In [ ]:
def ravine(p):
    x, y = p
    return 0.5 * x**2 + 10 * y**2

def run(make_opt, steps=60, start=(-8.0, 1.0)):
    """ optimize the ravine from `start`; return the trajectory of (x, y) points """
    params = [Value(start[0]), Value(start[1])]
    opt = make_opt(params)
    path = [(params[0].data, params[1].data)]
    for _ in range(steps):
        loss = ravine(params)
        opt.zero_grad()
        loss.backward()
        opt.step()
        path.append((params[0].data, params[1].data))
        if abs(params[0].data) > 1e6 or abs(params[1].data) > 1e6:
            break  # diverged
    return path

def plot_paths(paths, title=""):
    xs, ys = np.meshgrid(np.linspace(-9, 2, 200), np.linspace(-2, 2, 200))
    plt.figure(figsize=(9, 3.5))
    plt.contour(xs, ys, 0.5 * xs**2 + 10 * ys**2, levels=30, cmap='Greys', linewidths=0.5)
    for name, path in paths.items():
        px, py = zip(*path)
        plt.plot(px, py, '.-', label=name, markersize=3, linewidth=1)
    plt.plot(0, 0, 'k*', markersize=12)
    plt.xlim(-9, 2); plt.ylim(-2, 2); plt.legend(); plt.title(title)
    plt.show()

### Exercise 1.1 — SGD
Every optimizer here subclasses `nanograd.optim.Optimizer`, which stores `self.params` and `self.lr` and provides `zero_grad()`.
Implement `step()` for plain gradient descent: $p \leftarrow p - \text{lr}\cdot\nabla p$.

In [ ]:
class SGD(Optimizer):
    def __init__(self, params, lr=0.1):
        super().__init__(params, lr)

    def step(self):
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
def test_one_step(make_opt, expected):
    p = Value(1.0); p.grad = 0.5
    make_opt([p]).step()
    return abs(p.data - expected) < 1e-9

def test_converges(make_opt, steps=300):
    x, y = run(make_opt, steps)[-1]
    return abs(x) < 1e-2 and abs(y) < 1e-2

check("SGD single step", lambda: test_one_step(lambda ps: SGD(ps, lr=0.1), 0.95))
check("SGD minimizes the ravine", lambda: test_converges(lambda ps: SGD(ps, lr=0.04)))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
class SGD(Optimizer):
    def __init__(self, params, lr=0.1):
        super().__init__(params, lr)

    def step(self):
        for p in self.params:
            p.data -= self.lr * p.grad

check("SGD single step", lambda: test_one_step(lambda ps: SGD(ps, lr=0.1), 0.95))
check("SGD minimizes the ravine", lambda: test_converges(lambda ps: SGD(ps, lr=0.04)))

In [ ]:
plot_paths({f"SGD lr={lr}": run(lambda ps, lr=lr: SGD(ps, lr=lr)) for lr in (0.02, 0.09)}, "plain SGD on the ravine")

With a small learning rate SGD crawls along the flat `x` direction. With a large one it makes faster progress in `x`, but
**zig-zags** across the steep `y` direction. We can't raise the learning rate further to speed up `x` — let's find out why.

### Exercise 1.2 — the largest stable learning rate
Run SGD for a few learning rates and look at whether it converges. Then fill in `max_stable_lr` with the largest
learning rate (to 2 decimal places) for which SGD still converges on this ravine. Bonus: explain it with a formula.

*Hint: along `y` the update is $y \leftarrow y - \text{lr}\cdot 20y = (1 - 20\,\text{lr})\,y$. When does that shrink?*

In [ ]:
for lr in (0.05, 0.09, 0.099, 0.101, 0.11):
    x, y = run(lambda ps: SGD(ps, lr=lr), steps=200)[-1]
    print(f"lr={lr:<6}  final y = {y:.3g}")

In [ ]:
max_stable_lr = None  # YOUR ANSWER HERE

In [ ]:
def test_max_lr():
    if max_stable_lr is None:
        raise NotImplementedError
    return abs(max_stable_lr - 0.1) < 0.005

check("max stable lr", test_max_lr)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
max_stable_lr = 2 / 20   # = 2 / (largest curvature). |1 - 20 lr| < 1  <=>  lr < 0.1
check("max stable lr", test_max_lr)

**Takeaway:** the steepest direction caps the learning rate at $2/\lambda_{max}$ (the curvature), and the flattest direction then
moves painfully slowly. The ratio between the two (here 20) is called the *condition number*. Real networks have huge
condition numbers — which is what momentum and Adam are designed for.

### Exercise 1.3 — Momentum
Keep a **velocity** for every parameter: a running sum of past gradients that decays by `momentum` each step.

$$v \leftarrow \mu\, v + \nabla p, \qquad p \leftarrow p - \text{lr}\cdot v$$

In the zig-zag direction successive gradients have alternating signs and cancel out; in the consistent `x` direction they add up,
so the optimizer speeds up.

In [ ]:
class Momentum(Optimizer):
    def __init__(self, params, lr=0.1, momentum=0.9):
        super().__init__(params, lr)
        self.momentum = momentum
        # YOUR CODE HERE: initialize one velocity per parameter

    def step(self):
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
def test_momentum_two_steps():
    p = Value(1.0)
    opt = Momentum([p], lr=0.1, momentum=0.9)
    p.grad = 1.0; opt.step()   # v = 1.0,          p = 1.0 - 0.1     = 0.9
    p.grad = 1.0; opt.step()   # v = 0.9 + 1 = 1.9, p = 0.9 - 0.19    = 0.71
    return abs(p.data - 0.71) < 1e-9

check("Momentum update rule", test_momentum_two_steps)
check("Momentum minimizes the ravine", lambda: test_converges(lambda ps: Momentum(ps, lr=0.02, momentum=0.9)))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
class Momentum(Optimizer):
    def __init__(self, params, lr=0.1, momentum=0.9):
        super().__init__(params, lr)
        self.momentum = momentum
        self.velocity = [0.0 for _ in self.params]

    def step(self):
        for i, p in enumerate(self.params):
            self.velocity[i] = self.momentum * self.velocity[i] + p.grad
            p.data -= self.lr * self.velocity[i]

check("Momentum update rule", test_momentum_two_steps)
check("Momentum minimizes the ravine", lambda: test_converges(lambda ps: Momentum(ps, lr=0.02, momentum=0.9)))

### Exercise 1.4 — Adam
Adam keeps two running averages per parameter:

* **m**, the average gradient (like momentum): $m \leftarrow \beta_1 m + (1-\beta_1)\, g$
* **v**, the average *squared* gradient: $v \leftarrow \beta_2 v + (1-\beta_2)\, g^2$

Both start at 0, so early on they are biased toward 0. Correct for it with the step count `t` (starting at 1):
$\hat m = m / (1-\beta_1^t)$, $\hat v = v / (1-\beta_2^t)$. Then

$$p \leftarrow p - \text{lr}\cdot \frac{\hat m}{\sqrt{\hat v} + \epsilon}$$

Dividing by $\sqrt{\hat v}$ gives every parameter its **own effective step size**: steep directions (big gradients) take
smaller steps and flat directions take bigger ones — which is exactly what the ravine needs.

In [ ]:
class Adam(Optimizer):
    def __init__(self, params, lr=0.01, betas=(0.9, 0.999), eps=1e-8):
        super().__init__(params, lr)
        self.beta1, self.beta2 = betas
        self.eps = eps
        # YOUR CODE HERE: initialize m, v and the step counter t

    def step(self):
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
def test_adam_first_step():
    # thanks to bias correction, the first step is ~lr no matter how big or small the gradient is
    for g in (1e-3, 1.0, 1e3):
        p = Value(0.0); p.grad = g
        Adam([p], lr=0.01).step()
        if abs(p.data + 0.01) > 1e-6:
            return False
    return True

check("Adam first step is lr-sized", test_adam_first_step)
check("Adam minimizes the ravine", lambda: test_converges(lambda ps: Adam(ps, lr=0.3), steps=500))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
class Adam(Optimizer):
    def __init__(self, params, lr=0.01, betas=(0.9, 0.999), eps=1e-8):
        super().__init__(params, lr)
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.m = [0.0 for _ in self.params]
        self.v = [0.0 for _ in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            g = p.grad
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g * g
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            p.data -= self.lr * m_hat / (math.sqrt(v_hat) + self.eps)

check("Adam first step is lr-sized", test_adam_first_step)
check("Adam minimizes the ravine", lambda: test_converges(lambda ps: Adam(ps, lr=0.3), steps=500))

In [ ]:
plot_paths({
    "SGD lr=0.09":                 run(lambda ps: SGD(ps, lr=0.09)),
    "Momentum lr=0.02, mu=0.9":    run(lambda ps: Momentum(ps, lr=0.02, momentum=0.9)),
    "Adam lr=0.5":                 run(lambda ps: Adam(ps, lr=0.5)),
}, "60 steps on the ravine")

Look at the shapes:
* **SGD** at nearly its largest stable learning rate wastes its first ~15 steps zig-zagging across `y`. It does get there, but it is
  balanced on a knife edge: at `lr=0.101` it would diverge.
* **Momentum** uses a 4–5× smaller learning rate, yet covers the `x` distance just as fast because the velocity builds up. The `y`
  oscillation is damped quickly, but the heavy ball **overshoots** the minimum and has to swing back.
* **Adam** never zig-zags: its very first step is diagonal, because after bias correction *both* coordinates move by ≈ `lr` whatever the
  size of their gradient. It then moves at a roughly constant speed set by `lr`, overshoots a little, and settles.

Try changing the learning rates and the number of steps — e.g. what is the best `lr` for each optimizer if you only get 20 steps?

## Part 2 — Racing optimizers on a real network

Toy surfaces build intuition, but let's check it on a neural net. We train the same MLP on the moons dataset with the same
initial weights, changing only the optimizer. We use the library versions from `nanograd.optim` (identical to yours;
`nanograd.optim.SGD` takes a `momentum=` argument instead of having a separate class).

In [ ]:
from nanograd import optim
from nanograd.nn import MLP
from nanograd.datasets import make_moons
from nanograd.losses import hinge
from nanograd.viz import plot_decision_boundary

X, y = make_moons(n_samples=60, noise=0.1, seed=0)
y_pm = [2 * t - 1 for t in y]   # hinge loss wants labels in {-1, +1}

def train(make_opt, steps=80, seed=1337):
    random.seed(seed)                    # identical starting weights for every optimizer
    model = MLP(2, [8, 8, 1])
    opt = make_opt(model.parameters())
    history = []
    for k in range(steps):
        scores = [model([Value(xi) for xi in x]) for x in X]
        loss = sum(hinge(s, t) for s, t in zip(scores, y_pm)) * (1.0 / len(X))
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(loss.data)
    return model, history

contenders = {
    "SGD lr=0.1":            lambda ps: optim.SGD(ps, lr=0.1),
    "Momentum lr=0.1 mu=0.9": lambda ps: optim.SGD(ps, lr=0.1, momentum=0.9),
    "Adam lr=0.05":          lambda ps: optim.Adam(ps, lr=0.05),
}
results = {name: train(make) for name, make in contenders.items()}

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for name, (model, hist) in results.items():
    axes[0].plot(hist, label=name)
axes[0].set_yscale('log'); axes[0].set_xlabel('step'); axes[0].set_title('training loss (log scale)'); axes[0].legend()
for ax, (name, (model, hist)) in zip(axes[1:], results.items()):
    plot_decision_boundary(model, X, y, ax=ax, h=0.1, title=f"{name}\nfinal loss {hist[-1]:.3f}")
plt.tight_layout(); plt.show()

### Exercise 2.1 — a learning-rate schedule
`demo.ipynb` decays the learning rate linearly from 1.0 to 0.1 over training. A schedule lets us take big steps early and small,
careful steps near the end. Implement `linear_decay` and use it to set `opt.lr` before every step.

In [ ]:
def linear_decay(step, total_steps, lr_start, lr_end):
    """ lr_start at step 0, lr_end at step total_steps, linear in between """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check("linear_decay endpoints", lambda: linear_decay(0, 100, 1.0, 0.1) == 1.0 and abs(linear_decay(100, 100, 1.0, 0.1) - 0.1) < 1e-12)
check("linear_decay midpoint", lambda: abs(linear_decay(50, 100, 1.0, 0.1) - 0.55) < 1e-12)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def linear_decay(step, total_steps, lr_start, lr_end):
    frac = step / total_steps
    return lr_start + frac * (lr_end - lr_start)

check("linear_decay endpoints", lambda: linear_decay(0, 100, 1.0, 0.1) == 1.0 and abs(linear_decay(100, 100, 1.0, 0.1) - 0.1) < 1e-12)
check("linear_decay midpoint", lambda: abs(linear_decay(50, 100, 1.0, 0.1) - 0.55) < 1e-12)

In [ ]:
class ScheduledSGD(optim.SGD):
    """ SGD whose lr follows linear_decay; the schedule just mutates opt.lr before each step """
    def __init__(self, params, total_steps=80, lr_start=1.0, lr_end=0.1):
        super().__init__(params, lr=lr_start)
        self.k, self.total, self.lr_start, self.lr_end = 0, total_steps, lr_start, lr_end
    def step(self):
        self.lr = linear_decay(self.k, self.total, self.lr_start, self.lr_end)
        super().step()
        self.k += 1

_, hist_sched = train(lambda ps: ScheduledSGD(ps))
_, hist_const = train(lambda ps: optim.SGD(ps, lr=1.0))
plt.figure(figsize=(5, 3))
plt.plot(results["SGD lr=0.1"][1], label="constant lr=0.1")
plt.plot(hist_const, label="constant lr=1.0")
plt.plot(hist_sched, label="schedule lr 1.0 -> 0.1")
plt.yscale('log'); plt.legend(); plt.xlabel('step'); plt.title('effect of a schedule'); plt.show()

An honest result: on this tiny problem the schedule does **not** beat a well-chosen constant learning rate. Our gradients
are exact (we use the whole dataset every step), so a big constant step keeps making progress all the way down. Schedules earn their
keep with **mini-batches**: each step then sees a noisy gradient estimate, a large constant `lr` keeps bouncing around the
minimum, and shrinking it lets the noise average out. Try it: change `train` to use a random batch of 10 points per step and rerun.

### Exercise 2.2 — experiment (open-ended)
Try to break each optimizer: what happens to SGD with `lr=2.0`? To Adam with `lr=1.0`? To Momentum with `momentum=0.99`?
Write down what you observe and why.

In [ ]:
# YOUR CODE HERE

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
for name, make in {"SGD lr=2.0": lambda ps: optim.SGD(ps, lr=2.0),
                   "Adam lr=1.0": lambda ps: optim.Adam(ps, lr=1.0),
                   "Momentum mu=0.99": lambda ps: optim.SGD(ps, lr=0.1, momentum=0.99)}.items():
    _, h = train(make, steps=40)
    plt.plot(h, label=name)
plt.yscale('log'); plt.legend(); plt.xlabel('step'); plt.show()
# What to look for (exact curves depend on the seed):
# - SGD lr=2.0: the loss jumps UP at times - each step overshoots the valley. Big steps can also
#   kill ReLUs for good (Lesson 4).
# - Adam lr=1.0: every parameter moves by roughly lr per step whatever its gradient, so it keeps
#   bouncing and ends up worst of the three.
# - momentum=0.99: it remembers ~100 steps of history, so it keeps rolling in old directions
#   and its curve is bumpier than with 0.9.

## Summary
| optimizer | memory per parameter | idea |
|---|---|---|
| SGD | none | step downhill, step size fixed by `lr` (capped at 2 / curvature) |
| Momentum | velocity | average away zig-zags, speed up in consistent directions |
| Adam | m and v (2 numbers) | momentum + per-parameter step size, with bias correction |

The optimizer never looks at the model or the graph — only at `p.data` and `p.grad`. That clean separation is why
PyTorch can offer `torch.optim.Adam(model.parameters())` for *any* model.